# Average age and gender counts

This notebook reads `all_survey_rows.csv`, extracts the demographics survey rows, and calculates:

- average age
- minimum age
- maximum age
- total female participants
- total male participants

It keeps the **first** age/gender entry per participant in case there are duplicates.


In [1]:

import pandas as pd
import numpy as np

# file path
csv_path = "all_survey_rows.csv"

# load data
df = pd.read_csv(csv_path)

# keep only demographics rows for age and gender
demo = df[
    (df["survey_name"] == "post_survey_demographics") &
    (df["item_id"].isin(["age", "gender"]))
].copy()

# sort so we can keep the first entry per participant/item
demo["source_row_line_num"] = pd.to_numeric(demo["source_row_line"], errors="coerce")
demo = demo.sort_values(["participant_id", "item_id", "source_row_line_num"])

# keep the first age/gender value per participant
demo = demo.drop_duplicates(["participant_id", "item_id"], keep="first")

# reshape to one row per participant
demo_wide = demo.pivot(index="participant_id", columns="item_id", values="raw_value").reset_index()

demo_wide.head()


item_id,participant_id,age,gender
0,55bb9ae7fdf99b26d27fda01,39,female
1,56514c84715ff50011a43878,39,female
2,5cb9ca6814b3cb0017e4f7dd,38,Male
3,5d4c78071b920f00198e8533,56,female
4,5e9cbc3999531a099492509f,26,male


In [2]:

# clean age
demo_wide["age_num"] = pd.to_numeric(demo_wide["age"], errors="coerce")

# normalize gender labels
def normalize_gender(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = s.replace("_", " ").replace("-", " ")

    if "female" in s or "woman" in s:
        return "female"
    if "male" in s or "man" in s:
        return "male"

    return s

demo_wide["gender_norm"] = demo_wide["gender"].apply(normalize_gender)

demo_wide[["participant_id", "age", "age_num", "gender", "gender_norm"]].head()


item_id,participant_id,age,age_num,gender,gender_norm
0,55bb9ae7fdf99b26d27fda01,39,39,female,female
1,56514c84715ff50011a43878,39,39,female,female
2,5cb9ca6814b3cb0017e4f7dd,38,38,Male,male
3,5d4c78071b920f00198e8533,56,56,female,female
4,5e9cbc3999531a099492509f,26,26,male,male


In [3]:
# summary
average_age = demo_wide["age_num"].mean()
min_age = demo_wide["age_num"].min()
max_age = demo_wide["age_num"].max()
female_total = (demo_wide["gender_norm"] == "female").sum()
male_total = (demo_wide["gender_norm"] == "male").sum()
total_participants = demo_wide["participant_id"].nunique()

print(f"Total participants: {total_participants}")
print(f"Average age: {average_age:.2f}")
print(f"Minimum age: {min_age:.0f}" if pd.notna(min_age) else "Minimum age: NA")
print(f"Maximum age: {max_age:.0f}" if pd.notna(max_age) else "Maximum age: NA")
print(f"Total female participants: {female_total}")
print(f"Total male participants: {male_total}")


Total participants: 49
Average age: 39.59
Minimum age: 15
Maximum age: 66
Total female participants: 27
Total male participants: 22


In [4]:

# optional: save participant-level demographics table
output_path = "participant_age_gender_summary.csv"
demo_wide.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


Saved: participant_age_gender_summary.csv
